# Module 07 — GPU Bottlenecks and the Roofline Model

This notebook measures memory-oriented vector addition, FP16 GEMM, and projection shapes on a Lightning NVIDIA T4. The goal is to form a bottleneck hypothesis, not to chase one peak number.

## Lab protocol

Warm up kernels, keep tensors on the GPU, use CUDA events, report medians, and record shape, dtype, work, and memory state with every result.

In [ ]:
# Standalone implementation: this notebook does not import roofline_benchmarks.py.
import statistics
from dataclasses import dataclass

import torch

T4_FP32_TFLOPS, T4_FP16_TFLOPS, T4_BANDWIDTH_GBS = 8.1, 65.0, 300.0

@dataclass
class Measurement:
    label: str
    milliseconds: float
    tflops: float
    arithmetic_intensity: float
    effective_bandwidth_gbs: float = 0.0

assert torch.cuda.is_available(), 'Select a CUDA-enabled T4 runtime.'
device = torch.device('cuda:0')

def gpu_benchmark(fn, warmup=15, repeats=50):
    for _ in range(warmup): fn()
    torch.cuda.synchronize(device)
    samples = []
    for _ in range(repeats):
        start = torch.cuda.Event(enable_timing=True); end = torch.cuda.Event(enable_timing=True)
        start.record(); fn(); end.record(); end.synchronize()
        samples.append(start.elapsed_time(end))
    return statistics.median(samples)

def print_roofline_summary():
    print('FP32 balance:', T4_FP32_TFLOPS * 1000 / T4_BANDWIDTH_GBS, 'FLOPs/byte')
    print('FP16 Tensor Core balance:', T4_FP16_TFLOPS * 1000 / T4_BANDWIDTH_GBS, 'FLOPs/byte')
    print('Peaks are theoretical ceilings, not measured application results.')

def vector_add_experiment(device, elements=16_000_000, warmup=15, repeats=50):
    a = torch.randn(elements, device=device, dtype=torch.float32); b = torch.randn_like(a); c = torch.empty_like(a)
    ms = gpu_benchmark(lambda: torch.add(a, b, out=c), warmup, repeats)
    useful_bytes = 3 * elements * 4
    result = Measurement('vector add', ms, elements / (ms / 1000) / 1e12, 1 / 12, useful_bytes / (ms / 1000) / 1e9)
    print(result); return result

def gemm_experiment(device, size=4096, warmup=15, repeats=50):
    a = torch.randn((size, size), device=device, dtype=torch.float16); b = torch.randn_like(a); c = torch.empty_like(a)
    ms = gpu_benchmark(lambda: torch.mm(a, b, out=c), warmup, repeats)
    flops = 2 * size**3; ai = flops / ((a.numel() + b.numel() + c.numel()) * 2)
    result = Measurement('FP16 GEMM', ms, flops / (ms / 1000) / 1e12, ai)
    print(result); return result

def projection_experiment(device, batches, hidden_size=4096, warmup=15, repeats=50):
    weight = torch.randn((hidden_size, hidden_size), device=device, dtype=torch.float16); results = []
    for batch in batches:
        x = torch.randn((batch, hidden_size), device=device, dtype=torch.float16); y = torch.empty_like(x)
        ms = gpu_benchmark(lambda: torch.mm(x, weight, out=y), warmup, repeats)
        flops = 2 * batch * hidden_size**2; bytes_min = (x.numel() + weight.numel() + y.numel()) * 2
        result = Measurement(f'Projection batch {batch}', ms, flops / (ms / 1000) / 1e12, flops / bytes_min)
        results.append(result); print(result)
    return results

print('Device:', device)
print_roofline_summary()

## 1. Memory-oriented vector addition

FP32 vector add performs one FLOP for 12 useful bytes, so its arithmetic intensity is approximately 0.083 FLOPs/byte.

In [ ]:
# Experiment 1: stream three large FP32 arrays through a vector addition.
elements = 16_000_000
a = torch.randn(elements, device=device, dtype=torch.float32)
b = torch.randn_like(a)
c = torch.empty_like(a)
milliseconds = gpu_benchmark(lambda: torch.add(a, b, out=c))
useful_bytes = 3 * elements * 4
print('time (ms):', milliseconds)
print('useful GB/s:', useful_bytes / (milliseconds / 1000) / 1e9)
print('arithmetic intensity:', 1 / 12, 'FLOPs/byte')

## 2. Compute-oriented FP16 GEMM

In [ ]:
# Experiment 2: reuse tiled FP16 weights in a large matrix multiplication.
size = 4096
a = torch.randn((size, size), device=device, dtype=torch.float16)
b = torch.randn_like(a)
c = torch.empty_like(a)
milliseconds = gpu_benchmark(lambda: torch.mm(a, b, out=c))
flops = 2 * size**3
print('time (ms):', milliseconds)
print('achieved TFLOPS:', flops / (milliseconds / 1000) / 1e12)
print('idealized arithmetic intensity:', flops / ((a.numel() + b.numel() + c.numel()) * 2))

## 3. Projection shape: GEMV-like to GEMM-like

Keep the weight matrix fixed and vary the number of input rows. Compare batch-one decode-like work with larger batched shapes.

In [ ]:
# Experiment 3: hold weights fixed and vary token rows from decode-like to prefill-like.
hidden_size = 4096
weight = torch.randn((hidden_size, hidden_size), device=device, dtype=torch.float16)
results = []
for batch in [1, 4, 16, 32, 64, 128, 256, 512]:
    x = torch.randn((batch, hidden_size), device=device, dtype=torch.float16)
    y = torch.empty_like(x)
    milliseconds = gpu_benchmark(lambda: torch.mm(x, weight, out=y), warmup=15, repeats=50)
    flops = 2 * batch * hidden_size**2
    results.append({'batch': batch, 'milliseconds': milliseconds, 'tflops': flops / (milliseconds / 1000) / 1e12})
print(results)

In [ ]:
import matplotlib.pyplot as plt

batches = [result['batch'] for result in results]
tflops = [result['tflops'] for result in results]
plt.figure(figsize=(8, 4))
plt.plot(batches, tflops, marker='o')
plt.xlabel('Input rows / token positions')
plt.ylabel('Achieved TFLOP/s')
plt.title('T4 projection efficiency versus batch size')
plt.grid(alpha=0.3)
plt.show()

## Interpretation

Identify one memory-oriented result, one compute-oriented result, and one shape- or launch-sensitive result. These are hypotheses to validate with profiling and an end-to-end serving workload.